# AEGES-Q: Classical Cryptography Benchmark

## Objective

This notebook evaluates classical cryptographic primitives suitable for
integration into the AEGES-Q security framework.

The experiments establish a measurable classical cryptography baseline
before introducing post-quantum cryptographic mechanisms.

The benchmark evaluates:

- Key generation latency
- Encryption latency
- Decryption latency
- Throughput
- Ciphertext overhead
- Repeated-operation performance

The results will be used to guide the production cryptographic layer
of AEGES-Q and provide a baseline for later comparison with
post-quantum cryptography.

In [ ]:
# imports and configs
import os
import time
import statistics
from pathlib import Path

import numpy as np
import pandas as pd

from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.asymmetric.x25519 import (
    X25519PrivateKey,
)
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd().resolve().parents[1]

print("Project root:", PROJECT_ROOT)
print("Classical cryptography environment ready.")

Project root: C:\Projects\Aeges-Q
Classical cryptography environment ready.


In [ ]:
import cryptography

print("Cryptography version:", cryptography.__version__)
print("AESGCM available:", AESGCM is not None)
print("X25519 available:", X25519PrivateKey is not None)
print("HKDF available:", HKDF is not None)

Cryptography version: 50.0.1
AESGCM available: True
X25519 available: True
HKDF available: True


## 1. AES-256-GCM Baseline

AES-256-GCM is used as the primary classical symmetric encryption
primitive for the AEGES-Q baseline.

GCM provides authenticated encryption, combining:

- Confidentiality
- Integrity
- Authentication

A 256-bit AES key is generated for the experiment.

The implementation will be benchmarked for:

- Key generation
- Encryption
- Decryption
- Throughput
- Ciphertext overhead

In [ ]:
# Generate a random 256-bit AES key
aes_key = AESGCM.generate_key(bit_length=256)

print("AES key length:", len(aes_key), "bytes")
print("AES key size:", len(aes_key) * 8, "bits")

AES key length: 32 bytes
AES key size: 256 bits


In [ ]:
# Representative network/session payload
plaintext = (
    b"AEGES-Q network security session payload. "
    b"This data represents protected application traffic "
    b"processed by the classical cryptographic layer."
)

print("Plaintext size:", len(plaintext), "bytes")
print("Plaintext:", plaintext.decode())

Plaintext size: 140 bytes
Plaintext: AEGES-Q network security session payload. This data represents protected application traffic processed by the classical cryptographic layer.


In [ ]:
# Initialize AES-GCM
aesgcm = AESGCM(aes_key)

# Generate a fresh 96-bit nonce
nonce = os.urandom(12)

# Optional authenticated metadata
associated_data = b"AEGES-Q|CLASSICAL|AES-256-GCM"

# Encrypt
ciphertext = aesgcm.encrypt(
    nonce,
    plaintext,
    associated_data
)

print("Plaintext size:", len(plaintext), "bytes")
print("Nonce size:", len(nonce), "bytes")
print("Ciphertext size:", len(ciphertext), "bytes")

# Decrypt
decrypted = aesgcm.decrypt(
    nonce,
    ciphertext,
    associated_data
)

print("\nDecryption successful:", decrypted == plaintext)
print("Decrypted data:", decrypted.decode())

Plaintext size: 140 bytes
Nonce size: 12 bytes
Ciphertext size: 156 bytes

Decryption successful: True
Decrypted data: AEGES-Q network security session payload. This data represents protected application traffic processed by the classical cryptographic layer.


### 1.1 Authentication and Tamper Detection

AES-256-GCM provides authenticated encryption.

To verify the integrity protection experimentally, the ciphertext will
be modified before decryption.

A correctly implemented AES-GCM operation should reject the modified
ciphertext rather than returning corrupted plaintext.

In [ ]:
from cryptography.exceptions import InvalidTag

# Create a modified copy of the ciphertext
tampered_ciphertext = bytearray(ciphertext)
tampered_ciphertext[0] ^= 1
tampered_ciphertext = bytes(tampered_ciphertext)

# Attempt decryption of tampered ciphertext
try:
    aesgcm.decrypt(
        nonce,
        tampered_ciphertext,
        associated_data
    )
    print("Tampering detection: FAILED")

except InvalidTag:
    print("Tampering detection: PASSED")
    print("AES-GCM rejected the modified ciphertext.")

Tampering detection: PASSED
AES-GCM rejected the modified ciphertext.


## 2. AES-256-GCM Performance Benchmark

The benchmark measures AES-256-GCM performance across multiple payload sizes.

Metrics:

- Encryption latency
- Decryption latency
- Throughput
- Ciphertext overhead

Each operation uses a fresh nonce to maintain correct AES-GCM usage.

In [ ]:
# Benchmark configuration

PAYLOAD_SIZES = [
    1024,          # 1 KB
    4096,          # 4 KB
    16384,         # 16 KB
    65536,         # 64 KB
    1048576        # 1 MB
]

WARMUP_RUNS = 20
BENCHMARK_RUNS = 500

print("Payload sizes:", PAYLOAD_SIZES)
print("Warm-up runs:", WARMUP_RUNS)
print("Benchmark runs:", BENCHMARK_RUNS)

Payload sizes: [1024, 4096, 16384, 65536, 1048576]
Warm-up runs: 20
Benchmark runs: 500


In [ ]:
# AES-256-GCM performance benchmark

benchmark_results = []

for payload_size in PAYLOAD_SIZES:
    payload = os.urandom(payload_size)

    # Warm-up
    for _ in range(WARMUP_RUNS):
        nonce = os.urandom(12)
        encrypted = aesgcm.encrypt(nonce, payload, associated_data)
        aesgcm.decrypt(nonce, encrypted, associated_data)

    encryption_times = []
    decryption_times = []

    # Benchmark
    for _ in range(BENCHMARK_RUNS):
        nonce = os.urandom(12)

        # Encryption
        start = time.perf_counter()
        encrypted = aesgcm.encrypt(nonce, payload, associated_data)
        encryption_times.append(time.perf_counter() - start)

        # Decryption
        start = time.perf_counter()
        decrypted = aesgcm.decrypt(nonce, encrypted, associated_data)
        decryption_times.append(time.perf_counter() - start)

        # Safety check
        assert decrypted == payload

    encryption_median = statistics.median(encryption_times)
    decryption_median = statistics.median(decryption_times)

    encryption_mean = statistics.mean(encryption_times)
    decryption_mean = statistics.mean(decryption_times)

    encryption_throughput = payload_size / encryption_median / (1024 ** 2)
    decryption_throughput = payload_size / decryption_median / (1024 ** 2)

    benchmark_results.append({
        "payload_size_bytes": payload_size,
        "encryption_median_ms": encryption_median * 1000,
        "decryption_median_ms": decryption_median * 1000,
        "encryption_mean_ms": encryption_mean * 1000,
        "decryption_mean_ms": decryption_mean * 1000,
        "encryption_throughput_MBps": encryption_throughput,
        "decryption_throughput_MBps": decryption_throughput,
        "ciphertext_overhead_bytes": len(encrypted) - payload_size,
    })

benchmark_df = pd.DataFrame(benchmark_results)

benchmark_df

,payload_size_bytes,encryption_median_ms,decryption_median_ms,encryption_mean_ms,decryption_mean_ms,encryption_throughput_MBps,decryption_throughput_MBps,ciphertext_overhead_bytes
0,1024,0.00360,0.00360,0.003515,0.003468,271.267369,271.267369,16
1,4096,0.00480,0.00510,0.004991,0.005683,813.802055,765.931380,16
2,16384,0.01480,0.01440,0.016258,0.016918,1055.743249,1085.069440,16
3,65536,0.03450,0.03420,0.038215,0.037347,1811.594199,1827.485376,16
4,1048576,1.22165,1.19755,1.417238,1.367156,818.565056,835.038203,16


In [ ]:
# Format benchmark results for analysis

display_df = benchmark_df.copy()

display_df["payload_size_kb"] = (
    display_df["payload_size_bytes"] / 1024
)

display_df = display_df[
    [
        "payload_size_bytes",
        "payload_size_kb",
        "encryption_median_ms",
        "decryption_median_ms",
        "encryption_throughput_MBps",
        "decryption_throughput_MBps",
        "ciphertext_overhead_bytes",
    ]
]

display_df

,payload_size_bytes,payload_size_kb,encryption_median_ms,decryption_median_ms,encryption_throughput_MBps,decryption_throughput_MBps,ciphertext_overhead_bytes
0,1024,1.0,0.00360,0.00360,271.267369,271.267369,16
1,4096,4.0,0.00480,0.00510,813.802055,765.931380,16
2,16384,16.0,0.01480,0.01440,1055.743249,1085.069440,16
3,65536,64.0,0.03450,0.03420,1811.594199,1827.485376,16
4,1048576,1024.0,1.22165,1.19755,818.565056,835.038203,16


In [ ]:
# Calculate ciphertext overhead percentage

benchmark_df["overhead_percentage"] = (
    benchmark_df["ciphertext_overhead_bytes"]
    / benchmark_df["payload_size_bytes"]
    * 100
)

overhead_df = benchmark_df[
    [
        "payload_size_bytes",
        "ciphertext_overhead_bytes",
        "overhead_percentage"
    ]
].copy()

overhead_df

,payload_size_bytes,ciphertext_overhead_bytes,overhead_percentage
0,1024,16,1.562500
1,4096,16,0.390625
2,16384,16,0.097656
3,65536,16,0.024414
4,1048576,16,0.001526


In [ ]:
# Save AES-256-GCM benchmark results

crypto_artifacts_dir = PROJECT_ROOT / "artifacts" / "crypto"
crypto_artifacts_dir.mkdir(parents=True, exist_ok=True)

aes_benchmark_path = (
    crypto_artifacts_dir / "aes_256_gcm_benchmark.csv"
)

benchmark_df.to_csv(aes_benchmark_path, index=False)

print("Saved:", aes_benchmark_path)
print("Rows:", len(benchmark_df))

Saved: C:\Projects\Aeges-Q\artifacts\crypto\aes_256_gcm_benchmark.csv
Rows: 5


## 3. X25519 + HKDF Key Establishment

X25519 is used for ephemeral elliptic-curve Diffie-Hellman key agreement.

HKDF-SHA256 is then used to derive a fixed-length application/session key
from the shared secret.

The experiment verifies:

- X25519 key generation
- Shared-secret agreement
- HKDF session-key derivation
- Agreement between both parties
- Key-establishment latency

In [ ]:
# X25519 + HKDF configuration

HKDF_INFO = b"AEGES-Q|CLASSICAL|SESSION-KEY"
DERIVED_KEY_LENGTH = 32

print("Key agreement: X25519")
print("KDF: HKDF-SHA256")
print("Derived key length:", DERIVED_KEY_LENGTH, "bytes")

Key agreement: X25519
KDF: HKDF-SHA256
Derived key length: 32 bytes


In [ ]:
# Generate ephemeral X25519 key pairs for two parties

client_private_key = X25519PrivateKey.generate()
client_public_key = client_private_key.public_key()

server_private_key = X25519PrivateKey.generate()
server_public_key = server_private_key.public_key()

print("Client key pair generated.")
print("Server key pair generated.")

Client key pair generated.
Server key pair generated.


In [ ]:
# Perform X25519 Diffie-Hellman key agreement

client_shared_secret = client_private_key.exchange(server_public_key)
server_shared_secret = server_private_key.exchange(client_public_key)

print("Client shared secret length:", len(client_shared_secret), "bytes")
print("Server shared secret length:", len(server_shared_secret), "bytes")
print(
    "Shared secrets match:",
    client_shared_secret == server_shared_secret
)

Client shared secret length: 32 bytes
Server shared secret length: 32 bytes
Shared secrets match: True


In [ ]:
# Derive a 256-bit session key from the shared secret using HKDF-SHA256

client_session_key = HKDF(
    algorithm=hashes.SHA256(),
    length=DERIVED_KEY_LENGTH,
    salt=None,
    info=HKDF_INFO,
).derive(client_shared_secret)

server_session_key = HKDF(
    algorithm=hashes.SHA256(),
    length=DERIVED_KEY_LENGTH,
    salt=None,
    info=HKDF_INFO,
).derive(server_shared_secret)

print("Client session key length:", len(client_session_key), "bytes")
print("Server session key length:", len(server_session_key), "bytes")
print(
    "Derived session keys match:",
    client_session_key == server_session_key
)

Client session key length: 32 bytes
Server session key length: 32 bytes
Derived session keys match: True


In [ ]:
# X25519 + HKDF key-establishment benchmark

KEY_EXCHANGE_WARMUP_RUNS = 20
KEY_EXCHANGE_BENCHMARK_RUNS = 500

key_exchange_times = []

# Warm-up
for _ in range(KEY_EXCHANGE_WARMUP_RUNS):
    client_private = X25519PrivateKey.generate()
    client_public = client_private.public_key()

    server_private = X25519PrivateKey.generate()
    server_public = server_private.public_key()

    client_secret = client_private.exchange(server_public)
    server_secret = server_private.exchange(client_public)

    assert client_secret == server_secret

    client_key = HKDF(
        algorithm=hashes.SHA256(),
        length=DERIVED_KEY_LENGTH,
        salt=None,
        info=HKDF_INFO,
    ).derive(client_secret)

    server_key = HKDF(
        algorithm=hashes.SHA256(),
        length=DERIVED_KEY_LENGTH,
        salt=None,
        info=HKDF_INFO,
    ).derive(server_secret)

    assert client_key == server_key


# Benchmark
for _ in range(KEY_EXCHANGE_BENCHMARK_RUNS):
    start = time.perf_counter()

    # Generate ephemeral key pairs
    client_private = X25519PrivateKey.generate()
    client_public = client_private.public_key()

    server_private = X25519PrivateKey.generate()
    server_public = server_private.public_key()

    # X25519 key agreement
    client_secret = client_private.exchange(server_public)
    server_secret = server_private.exchange(client_public)

    # HKDF-SHA256 key derivation
    client_key = HKDF(
        algorithm=hashes.SHA256(),
        length=DERIVED_KEY_LENGTH,
        salt=None,
        info=HKDF_INFO,
    ).derive(client_secret)

    server_key = HKDF(
        algorithm=hashes.SHA256(),
        length=DERIVED_KEY_LENGTH,
        salt=None,
        info=HKDF_INFO,
    ).derive(server_secret)

    elapsed = time.perf_counter() - start

    assert client_secret == server_secret
    assert client_key == server_key

    key_exchange_times.append(elapsed)


key_exchange_median_ms = statistics.median(key_exchange_times) * 1000
key_exchange_mean_ms = statistics.mean(key_exchange_times) * 1000
key_exchange_std_ms = statistics.stdev(key_exchange_times) * 1000

print("X25519 + HKDF benchmark")
print("------------------------")
print(f"Runs: {KEY_EXCHANGE_BENCHMARK_RUNS}")
print(f"Median latency: {key_exchange_median_ms:.4f} ms")
print(f"Mean latency:   {key_exchange_mean_ms:.4f} ms")
print(f"Std deviation:  {key_exchange_std_ms:.4f} ms")
print("Session-key agreement verified: True")

X25519 + HKDF benchmark
------------------------
Runs: 500
Median latency: 0.3722 ms
Mean latency:   0.4171 ms
Std deviation:  0.1065 ms
Session-key agreement verified: True
